In [1]:
import random
import re
from collections import Counter, defaultdict
import math
import pandas as pd


In [2]:

with open("hindi_sentences.txt", "r", encoding="utf-8") as f:
    corpus = [line.strip().split() for line in f if line.strip()]

# Add <s>, </s> markers for sentence boundaries
sentences = [["<s>"] + sent + ["</s>"] for sent in corpus]

print("Total sentences:", len(sentences))
print("Example sentence:", sentences[0][:15])

Total sentences: 163148
Example sentence: ['<s>', 'लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम', '</s>']


In [3]:
random.shuffle(sentences)

validation_set = sentences[:1000]
test_set = sentences[1000:2000]
train_set = sentences[2000:10000]

print(f"Train: {len(train_set)}, Validation: {len(validation_set)}, Test: {len(test_set)}")


Train: 161148, Validation: 1000, Test: 1000


In [4]:
def build_ngram_counts(corpus, n):
    ngram_counts = Counter()
    context_counts = Counter()
    for sent in corpus:
        for i in range(len(sent) - n + 1):
            ngram = tuple(sent[i:i+n])
            context = tuple(sent[i:i+n-1]) if n > 1 else ()
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts

def n_gram(sentence, n):
   
    if len(sentence) < n:
        return []
    return [tuple(sentence[i:i+n]) for i in range(len(sentence)-n+1)]


unigram_counts, _ = build_ngram_counts(train_set, 1)
bigram_counts, bigram_contexts = build_ngram_counts(train_set, 2)
trigram_counts, trigram_contexts = build_ngram_counts(train_set, 3)
quadgram_counts, quad_contexts = build_ngram_counts(train_set, 4)

In [ ]:
type(unigram_counts)

In [5]:
import pandas as pd

def good_turing_discount(counts, max_freq=100):
    Nc = Counter(counts.values())
    freq_table = []
    adjusted_counts = {}
    
    for c in range(max_freq+1):
        Nc_val = Nc.get(c, 0)
        Nc1_val = Nc.get(c+1, 0)
        if Nc_val > 0:
            c_star = (c+1) * Nc1_val / Nc_val if Nc1_val > 0 else c
            freq_table.append([c, Nc_val, c_star])
        else:
            freq_table.append([c, 0, 0])
    
    # Apply discount
    for ngram, c in counts.items():
        Nc_val = Nc[c]
        Nc1_val = Nc.get(c+1, 0)
        if Nc1_val > 0:
            c_star = (c+1) * Nc1_val / Nc_val
            adjusted_counts[ngram] = c_star
        else:
            adjusted_counts[ngram] = c
    
    df = pd.DataFrame(freq_table, columns=["C (MLE)", "Nc", "C*"])
    return adjusted_counts, df


In [6]:
uni_gt, uni_freq_table = good_turing_discount(unigram_counts)
bi_gt, bi_freq_table = good_turing_discount(bigram_counts)
tri_gt, tri_freq_table = good_turing_discount(trigram_counts)
quad_gt, quad_freq_table = good_turing_discount(quadgram_counts)

# Show top 100 frequencies for unigrams
uni_freq_table.head(100)


,C (MLE),Nc,C*
0,0,0,0.000000
1,1,85411,0.414818
2,2,17715,1.378662
3,3,8141,2.357941
4,4,4799,3.396541
...,...,...,...
95,95,29,102.620690
96,96,31,75.096774
97,97,24,147.000000
98,98,36,82.500000


In [ ]:
def get_prob(ngram, counts, lower_counts):
    """
    Compute relative frequency for an ngram given counts.
    Example: for trigram (w1,w2,w3), 
    P = count(w1,w2,w3) / count(w1,w2)
    """
    c_ng = counts.get(ngram, 0)
    if len(ngram) == 1:
        total = sum(counts.values())
    else:
        prefix = ngram[:-1]
        total = sum(v for k,v in counts.items() if k[:-1] == prefix)
    return c_ng / total if total > 0 else 0


def deleted_interpolation(train_set):
    # Build n-gram counts
    uni,_ = build_ngram_counts(train_set, 1)
    bi,_ = build_ngram_counts(train_set, 2)
    tri,_ = build_ngram_counts(train_set, 3)
    quad,_ = build_ngram_counts(train_set, 4)

    # Initialize λ counters
    lambda_counts = [0.0, 0.0, 0.0, 0.0]

    for sent in train_set:
        quadgrams = n_gram(sent, 4)
        for qg in quadgrams:
            w1,w2,w3,w4 = qg

            # For quadrigram
            p4 = (quad.get(qg,0)-1) / (tri.get((w1,w2,w3),0)) if tri.get((w1,w2,w3),0) > 0 else 0
            # For trigram
            p3 = (tri.get((w2,w3,w4),0)-1) / (bi.get((w2,w3),0)) if bi.get((w2,w3),0) > 0 else 0
            # For bigram
            p2 = (bi.get((w3,w4),0)-1) / (uni.get((w3,),0)) if uni.get((w3,),0) > 0 else 0
            # For unigram
            p1 = (uni.get((w4,),0)-1) / sum(uni.values())

            probs = [p1, p2, p3, p4]
            best = probs.index(max(probs))   # which model predicts best
            lambda_counts[best] += 1

    # Normalize to get λ’s
    total = sum(lambda_counts)
    lambdas = [c/total for c in lambda_counts]
    return lambdas

lambdas = deleted_interpolation(train_set)
print("Interpolation weights:", lambdas)


In [ ]:
def sentence_probability_gt(sentence, counts, total_count, n):
    """
    Compute probability of a sentence using Good–Turing smoothed counts.
    """
    ngrams = n_gram(sentence, n)
    prob = 1.0
    for ng in ngrams:
        prob *= (counts.get(ng, 1) / total_count)   # unseen gets smoothed prob
    return prob


def evaluate_model_gt(sentences, counts, n):
    total_count = sum(counts.values())
    probs = []
    for sent in sentences:
        probs.append(sentence_probability_gt(sent, counts, total_count, n))
    return probs


# Probabilities for validation & test sets
val_probs_uni = evaluate_model_gt(validation_set, uni_gt, 1)
val_probs_bi = evaluate_model_gt(validation_set, bi_gt, 2)
val_probs_tri = evaluate_model_gt(validation_set, tri_gt, 3)
val_probs_quad = evaluate_model_gt(validation_set, quad_gt, 4)

test_probs_uni = evaluate_model_gt(test_set, uni_gt, 1)
test_probs_bi = evaluate_model_gt(test_set, bi_gt, 2)
test_probs_tri = evaluate_model_gt(test_set, tri_gt, 3)
test_probs_quad = evaluate_model_gt(test_set, quad_gt, 4)

print("Sample validation unigram prob:", val_probs_uni[0])
print("Sample test trigram prob:", test_probs_tri[0])


In [ ]:
def interpolated_prob(sentence, lambdas, uni, bi, tri, quad):
    """
    Compute probability of a sentence using deleted interpolation weights.
    """
    ngrams = n_gram(sentence, 4)   # use quadrigrams (covers up to 4-grams)
    prob = 1.0
    total_unigrams = sum(uni.values())

    for qg in ngrams:
        w1,w2,w3,w4 = qg

        # Unigram probability
        p1 = uni.get((w4,), 0) / total_unigrams if total_unigrams > 0 else 0

        # Bigram probability
        p2 = bi.get((w3,w4),0) / uni.get((w3,),1) if uni.get((w3,),0) > 0 else 0

        # Trigram probability
        p3 = tri.get((w2,w3,w4),0) / bi.get((w2,w3),1) if bi.get((w2,w3),0) > 0 else 0

        # Quadrigram probability
        p4 = quad.get(qg,0) / tri.get((w1,w2,w3),1) if tri.get((w1,w2,w3),0) > 0 else 0

        # Interpolated probability
        p = lambdas[0]*p1 + lambdas[1]*p2 + lambdas[2]*p3 + lambdas[3]*p4

        prob *= p if p > 0 else 1e-10   # tiny prob to avoid underflow

    return prob


def evaluate_interpolated(sentences, lambdas, uni, bi, tri, quad):
    """
    Evaluate a set of sentences with deleted interpolation.
    Returns list of probabilities for each sentence.
    """
    probs = []
    for sent in sentences:
        probs.append(interpolated_prob(sent, lambdas, uni, bi, tri, quad))
    return probs


In [ ]:
# Validation set
val_probs_interp = evaluate_interpolated(
    validation_set, lambdas,
    unigram_counts, bigram_counts, trigram_counts, quadgram_counts
)

# Test set
test_probs_interp = evaluate_interpolated(
    test_set, lambdas,
    unigram_counts, bigram_counts, trigram_counts, quadgram_counts
)

print("Sample validation (interpolated) prob:", val_probs_interp[0])
print("Sample test (interpolated) prob:", test_probs_interp[0])
